# Imports

In [61]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime
import pytrendy as pt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error, r2_score

In [62]:
df = pd.read_csv("test/pjme-hourly-6b036af3b74fd917577561.csv")
df

,Datetime,PJME_MW
0,2002-12-31 01:00:00,26498.0
1,2002-12-31 02:00:00,25147.0
2,2002-12-31 03:00:00,24574.0
3,2002-12-31 04:00:00,24393.0
4,2002-12-31 05:00:00,24860.0
...,...,...
145361,2018-01-01 20:00:00,44284.0
145362,2018-01-01 21:00:00,43751.0
145363,2018-01-01 22:00:00,42402.0
145364,2018-01-01 23:00:00,40164.0


In [63]:
df["Datetime"] = pd.to_datetime(df["Datetime"], format="%Y-%m-%d %H:%M:%S")
df = df.sort_values(by=["Datetime"])

In [64]:
df.info()
df.describe()
print(df.shape)
print(df.isnull().sum())
df.head()

<class 'pandas.DataFrame'>
Index: 145366 entries, 8734 to 140254
Data columns (total 2 columns):
 #   Column    Non-Null Count   Dtype         
---  ------    --------------   -----         
 0   Datetime  145366 non-null  datetime64[us]
 1   PJME_MW   145366 non-null  float64       
dtypes: datetime64[us](1), float64(1)
memory usage: 3.3 MB
(145366, 2)
Datetime    0
PJME_MW     0
dtype: int64


,Datetime,PJME_MW
8734,2002-01-01 01:00:00,30393.0
8735,2002-01-01 02:00:00,29265.0
8736,2002-01-01 03:00:00,28357.0
8737,2002-01-01 04:00:00,27899.0
8738,2002-01-01 05:00:00,28057.0


In [65]:
temp = df.groupby([df["Datetime"].dt.date]).count()
temp["Datetime"].unique()

array([23, 24, 25,  1])

In [66]:
temp[temp["Datetime"] == 25]

,Datetime,PJME_MW
Datetime,,
2014-11-02,25,25
2015-11-01,25,25
2016-11-06,25,25
2017-11-05,25,25


In [67]:
df = df.set_index("Datetime")

In [68]:
# options = ["2014-11-02", "2015-11-01", "2016-11-06", "2017-11-05"]
# df = df.loc[df.index.normalize().isin(pd.to_datetime(options))]
# df = df[~df.index.duplicated(keep="first")]
df = df[~df.index.duplicated(keep="first")]

In [69]:
df

,PJME_MW
Datetime,
2002-01-01 01:00:00,30393.0
2002-01-01 02:00:00,29265.0
2002-01-01 03:00:00,28357.0
2002-01-01 04:00:00,27899.0
2002-01-01 05:00:00,28057.0
...,...
2018-08-02 20:00:00,44057.0
2018-08-02 21:00:00,43256.0
2018-08-02 22:00:00,41552.0


In [70]:
dt = df.index.to_series()
gap = dt.shift(-1) - dt
print(gap.value_counts())

Datetime
0 days 01:00:00    145331
0 days 02:00:00        30
Name: count, dtype: int64


In [71]:
df.describe()
tempdf = df.reset_index()
tempdf.describe()

,Datetime,PJME_MW
count,145362,145362.000000
mean,2010-04-18 01:57:55.403475,32080.510780
min,2002-01-01 01:00:00,14544.000000
25%,2006-02-23 13:15:00,27573.000000
50%,2010-04-18 02:30:00,31421.000000
75%,2014-06-10 15:45:00,35650.000000
max,2018-08-03 00:00:00,62009.000000
std,NaN,6463.866507


In [72]:
idx_2h = gap[gap == pd.Timedelta("2 hours")].index
df.loc[idx_2h]
df
df.loc[idx_2h[1]]

PJME_MW    22823.0
Name: 2002-10-27 01:00:00, dtype: float64

In [73]:
peint

NameError: name 'peint' is not defined

In [ ]:
df.loc[idx_2h[1]]

In [74]:
for row in idx_2h:
    line = pd.DataFrame({"PJME_MW": [df.loc[row]["PJME_MW"],np.nan]}, index=[row,row + datetime.timedelta(hours=1)])
    df = pd.concat([df.loc[:row], line, df.loc[row + datetime.timedelta(hours=2):]])

In [75]:
df[df["PJME_MW"].isnull()].head()

,PJME_MW
2002-04-07 03:00:00,NaN
2002-10-27 02:00:00,NaN
2003-04-06 03:00:00,NaN
2003-10-26 02:00:00,NaN
2004-04-04 03:00:00,NaN


In [ ]:
dt = df.index.to_series()
gap = dt.shift(-1) - dt
print(gap.value_counts())

In [ ]:
df2 = df.copy(
    deep=True
)
df2 = df2.reset_index()
df2.columns = ["Datetime", "PJME_MW"]

In [ ]:
df2.to_csv("pjme.csv", index=False)

In [ ]:
plt.figure(figsize=(10,5))
sns.boxplot(df2["PJME_MW"])

In [ ]:
df2_monthly = (
    df2.assign(Datetime=pd.to_datetime(df2["Datetime"]))
    .resample("ME", on="Datetime")["PJME_MW"]
    .mean()
    .dropna()
    .reset_index()
)

results = pt.detect_trends(df2_monthly, date_col="Datetime", value_col="PJME_MW", plot=True)
results.print_summary()

# preprocessing

In [ ]:
df2_interp = (
    df2.assign(Datetime=pd.to_datetime(df2["Datetime"]))
    .sort_values("Datetime")
    .set_index("Datetime")
)

df2_interp["PJME_MW"] = df2_interp["PJME_MW"].interpolate(method="time")
df2_interp = df2_interp.reset_index()

In [ ]:
df2_interp.to_csv("valid_data.csv", index=False)

In [ ]:
df = df2_interp

In [ ]:
df.isna().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df_fe = df.copy()
df_fe["Datetime"] = pd.to_datetime(df_fe["Datetime"])

In [ ]:
df_fe = (
    df_fe.sort_values("Datetime")
    .groupby("Datetime", as_index=False)["PJME_MW"]
    .mean()
    .set_index("Datetime")
    .asfreq("h")
)

In [ ]:
df_fe["Hour"] = df_fe.index.hour
df_fe["DayOfWeek"] = df_fe.index.day_of_week
df_fe["Month"] = df_fe.index.month
df_fe["Year"] = df_fe.index.year
df_fe["IsWeekend"] = df_fe["DayOfWeek"].isin([5, 6]).astype(int)

In [ ]:
df_fe["lag_24"] = df_fe["PJME_MW"].shift(24)
df_fe["lag_168"] = df_fe["PJME_MW"].shift(168)

In [ ]:
df_fe["roll_24"] = df_fe["PJME_MW"].shift(1).rolling(window=24, min_periods=24).mean()

In [ ]:
df_fe_encoded = pd.get_dummies(
    df_fe.reset_index(),
    columns=["Hour", "DayOfWeek", "Month"],
    drop_first=True,
    dtype=int,
)

In [ ]:
df_fe_encoded = df_fe_encoded.dropna()

In [ ]:
X = df_fe_encoded.drop(columns=["PJME_MW", "Datetime"])
Y = df_fe_encoded["PJME_MW"]

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, shuffle=False)

# Modelisation

In [ ]:
target_col = "PJME_MW"
feature_cols = [
    "Hour",
    "DayOfWeek",
    "Month",
    "Year",
    "IsWeekend",
    "lag_24",
    "lag_168",
    "roll_24",
]

In [ ]:
lr_model = LinearRegression()
lr_model.fit(X_train, Y_train)

# Evaluation

In [ ]:
y_pred = lr_model.predict(X_test)
y_baseline = X_test["lag_24"]

In [ ]:
def evaluate_predictions(y_true, y_hat, name="Modèle"):
    mae = mean_absolute_error(y_true, y_hat)
    rmse = np.sqrt(mean_squared_error(y_true, y_hat))
    r2 = r2_score(y_true, y_hat)
    mape = mean_absolute_percentage_error(y_true, y_hat) * 100  # en %
    return {"Modèle": name, "MAE (MW)": mae, "RMSE (MW)": rmse, "R²": r2, "MAPE (%)": mape}

In [ ]:
metrics_df = pd.DataFrame([
    evaluate_predictions(Y_test, y_baseline, "Baseline (lag_24)"),
    evaluate_predictions(Y_test, y_pred, "Régression Linéaire"),
])
metrics_df::

# Azure in order

In [ ]:

# The script MUST contain a function named azureml_main
# which is the entry point for this module.

# imports up here can be used to
import pandas as pd

# The entry point function MUST have two input arguments.
# If the input port is not connected, the corresponding
# dataframe argument will be None.
#   Param<dataframe1>: a pandas.DataFrame
#   Param<dataframe2>: a pandas.DataFrame
def azureml_main(dataframe1 = None, dataframe2 = None):

    print(f"info: \n{dataframe1.info()}")
    print(f"describe: \n{dataframe1.describe()}")
    print(f"shape: \n{dataframe1.shape}")
    print(dataframe1.isnull().sum())
    print(dataframe1.head())



    return dataframe1,



In [ ]:

# The script MUST contain a function named azureml_main
# which is the entry point for this module.

# imports up here can be used to
import pandas as pd
import datetime
import numpy as np
# The entry point function MUST have two input arguments.
# If the input port is not connected, the corresponding
# dataframe argument will be None.
#   Param<dataframe1>: a pandas.DataFrame
#   Param<dataframe2>: a pandas.DataFrame
def azureml_main(dataframe1 = None, dataframe2 = None):

    dataframe1 = dataframe1.set_index("Datetime")
    dt = dataframe1.index.to_series()
    gap = dt.shift(-1) - dt

    idx_2h = gap[gap == pd.Timedelta("2 hours")].index

    for row in idx_2h:
        line = pd.DataFrame({"PJME_MW": [dataframe1.loc[row]["PJME_MW"],np.nan]}, index=[row,row + datetime.timedelta(hours=1)])
        dataframe1 = pd.concat([dataframe1.loc[:row], line, dataframe1.loc[row + datetime.timedelta(hours=2):]])


    df2 = dataframe1.copy(
        deep=True
    )
    df2 = df2.reset_index()
    print(df2.columns)
    df2.columns = ["Datetime", "PJME_MW"]


    df2_interp = (
        df2.assign(Datetime=pd.to_datetime(df2["Datetime"]))
        .sort_values("Datetime")
        .set_index("Datetime")
    )

    df2_interp["PJME_MW"] = df2_interp["PJME_MW"].interpolate(method="time")
    df2_interp = df2_interp.reset_index()

    return df2_interp,



In [ ]:

# The script MUST contain a function named azureml_main
# which is the entry point for this module.

# imports up here can be used to
import pandas as pd

# The entry point function MUST have two input arguments.
# If the input port is not connected, the corresponding
# dataframe argument will be None.
#   Param<dataframe1>: a pandas.DataFrame
#   Param<dataframe2>: a pandas.DataFrame
def azureml_main(dataframe1 = None, dataframe2 = None):

    df_fe = dataframe1.copy()
    df_fe["Datetime"] = pd.to_datetime(df_fe["Datetime"])

    df_fe = (
        df_fe.sort_values("Datetime")
        .groupby("Datetime", as_index=False)["PJME_MW"]
        .mean()
        .set_index("Datetime")
        .asfreq("h")
    )

    df_fe["Hour"] = df_fe.index.hour
    df_fe["DayOfWeek"] = df_fe.index.dayofweek
    df_fe["Month"] = df_fe.index.month
    df_fe["Year"] = df_fe.index.year
    df_fe["IsWeekend"] = df_fe["DayOfWeek"].isin([5, 6]).astype(int)

    df_fe["lag_24"] = df_fe["PJME_MW"].shift(24)
    df_fe["lag_168"] = df_fe["PJME_MW"].shift(168)

    df_fe["roll_24"] = df_fe["PJME_MW"].shift(1).rolling(window=24, min_periods=24).mean()

    df_fe_encoded = pd.get_dummies(
        df_fe.reset_index(),
        columns=["Hour", "DayOfWeek", "Month"],
        drop_first=True,
        dtype=int,
    )
    return df_fe_encoded,

